# CS344 - HW3: HDR Tone Mapping

## Descripción del problema

Una imagen HDR (High Dynamic Range) almacena valores de luminancia en punto flotante con un rango extremadamente amplio (0 a 275 en este caso). Si se mapea linealmente ese rango a [0, 255], el 98% de los píxeles quedan en negro porque la mayoría de valores están por debajo de 3.

**Solución:** redistribuir los valores de luminancia usando su distribución acumulada (CDF), técnica conocida como *histogram equalization*. Esto comprime el rango dinámico sin perder detalle visual.

## Pasos implementados en CUDA

| Paso | Operación | Patrón paralelo |
|------|-----------|----------------|
| 1 | Encontrar min y max de luminancia | Reducción paralela con shared memory |
| 2 | Calcular rango = max - min | Aritmética escalar |
| 3 | Construir histograma de luminancia | Map + scatter con `atomicAdd` |
| 4 | Exclusive prefix sum (scan) sobre histograma | Algoritmo de Blelloch |

> **Requisito:** Runtime → Change runtime type → **T4 GPU**

## Celda 1 — Configuración del entorno

Clonamos el repositorio del curso y instalamos `nvcc_plugin`, que permite compilar y ejecutar código CUDA directamente desde celdas de Colab.

In [ ]:
!git clone https://github.com/depctg/udacity-cs344-colab
!pip install git+https://github.com/depctg/nvcc4jupyter.git


## Celda 2 — Instalación de OpenCV

El proyecto usa OpenCV para cargar la imagen HDR (formato `.exr`) y guardar el resultado como PNG.
Instalamos `libopencv-dev` y verificamos la ruta de los headers para usarla en la compilación.

In [ ]:
!dpkg --configure -a
!apt-get install -y libopencv-dev
!find /usr -name 'opencv.hpp' 2>/dev/null | head -3


## Celda 3 — Parche de compatibilidad OpenCV 3 → 4

El código original fue escrito para OpenCV 3, donde las constantes se llamaban `CV_LOAD_IMAGE_COLOR`, `CV_BGR2RGBA`, etc.
En OpenCV 4 estas constantes fueron renombradas: `cv::IMREAD_COLOR`, `cv::COLOR_BGR2RGBA`.

Usamos `sed` para reemplazar automáticamente todas las ocurrencias en `loadSaveImage.cpp`.

In [ ]:
!sed -i 's/CV_LOAD_IMAGE_COLOR/cv::IMREAD_COLOR/g'     /content/udacity-cs344-colab/src/HW3/loadSaveImage.cpp
!sed -i 's/CV_LOAD_IMAGE_ANYDEPTH/cv::IMREAD_ANYDEPTH/g' /content/udacity-cs344-colab/src/HW3/loadSaveImage.cpp
!sed -i 's/CV_BGR2RGBA/cv::COLOR_BGR2RGBA/g'           /content/udacity-cs344-colab/src/HW3/loadSaveImage.cpp
!sed -i 's/CV_RGBA2BGR/cv::COLOR_RGBA2BGR/g'           /content/udacity-cs344-colab/src/HW3/loadSaveImage.cpp
print('Parche aplicado correctamente')


## Celda 4 — Implementación CUDA (`student_func.cu`)

Se implementan tres kernels CUDA y la función principal que los coordina.

### Kernel 1: Reducción paralela (min y max)
En lugar de recorrer secuencialmente los N píxeles, cada bloque reduce su porción en **shared memory** (memoria dentro del chip, ~100x más rápida que la global). La reducción toma `log₂(blockDim)` pasos.

```
Entrada:  [3, 1, 4, 1, 5, 9, 2, 6]
Paso 1:   [1, 1, 5, 2]     <- comparar pares
Paso 2:   [1, 2]           <- comparar pares
Paso 3:   [1]              <- min = 1
```

### Kernel 2: Histograma con `atomicAdd`
Cada thread calcula el bin de su píxel: `bin = (lum - min) / range * numBins`.
Como múltiples threads pueden caer en el mismo bin simultáneamente, se necesita `atomicAdd` para evitar condiciones de carrera.

### Kernel 3: Exclusive Scan de Blelloch
Convierte el histograma en CDF. El scan es **exclusive**: `cdf[i]` contiene la suma de bins anteriores **sin incluir** el bin `i`. Dos fases: up-sweep (sumas parciales) y down-sweep (propagación).

```
Histograma: [4, 7, 3, 2]
CDF:        [0, 4, 11, 14]   <- cdf[0] = 0 siempre
```

La celda usa `%%writefile` para guardar el código directamente al archivo `.cu`.

In [ ]:
%%writefile /content/udacity-cs344-colab/src/HW3/student_func.cu
#include "utils.h"
#include <float.h>
#include <stdio.h>

// =============================================================================
// KERNEL 1: Reduccion paralela para encontrar min y max de luminancia
//
// Cada bloque de threads reduce su porcion del arreglo en shared memory.
// Shared memory es ~100x mas rapida que memoria global porque esta en el chip.
// La reduccion toma log2(blockDim) pasos en lugar de N pasos secuenciales.
// =============================================================================
__global__
void reduce_min_max(const float* const d_in,
                    float* d_min_out,
                    float* d_max_out,
                    const size_t n)
{
    // Shared memory dinamica: primera mitad para min, segunda para max
    extern __shared__ float sdata[];
    float* s_min = sdata;
    float* s_max = sdata + blockDim.x;

    int tid = threadIdx.x;
    int idx = blockIdx.x * blockDim.x + threadIdx.x;

    // Cargar datos; si el thread esta fuera del rango usar valores neutros
    s_min[tid] = (idx < (int)n) ? d_in[idx] : FLT_MAX;
    s_max[tid] = (idx < (int)n) ? d_in[idx] : -FLT_MAX;
    __syncthreads();

    // Reduccion en arbol: cada paso, la mitad activa compara con s posiciones adelante
    for (int s = blockDim.x / 2; s > 0; s >>= 1) {
        if (tid < s) {
            s_min[tid] = min(s_min[tid], s_min[tid + s]);
            s_max[tid] = max(s_max[tid], s_max[tid + s]);
        }
        __syncthreads(); // Sincronizar antes del siguiente paso
    }

    // Thread 0 de cada bloque escribe el resultado parcial a memoria global
    if (tid == 0) {
        d_min_out[blockIdx.x] = s_min[0];
        d_max_out[blockIdx.x] = s_max[0];
    }
}

// =============================================================================
// KERNEL 2: Histograma de luminancia usando operaciones atomicas
//
// Sin atomicAdd, dos threads que caigan en el mismo bin simultaneamente
// generarian una condicion de carrera: ambos leeran el mismo valor,
// ambos sumaran 1, y uno sobreescribira al otro -> conteo incorrecto.
// atomicAdd garantiza que la suma sea correcta aunque serialize el acceso.
// =============================================================================
__global__
void histogram_kernel(const float* const d_logLuminance,
                      unsigned int* d_histo,
                      const float min_logLum,
                      const float range,
                      const size_t numBins,
                      const size_t n)
{
    int idx = blockIdx.x * blockDim.x + threadIdx.x;
    if (idx >= (int)n) return;

    // Formula del bin segun el enunciado del homework
    int bin = (int)((d_logLuminance[idx] - min_logLum) / range * numBins);

    // Clamp: si el valor es exactamente el maximo cae fuera del rango
    bin = min(bin, (int)numBins - 1);

    // Suma atomica: evita condicion de carrera entre threads del mismo bin
    atomicAdd(&d_histo[bin], 1u);
}

// =============================================================================
// KERNEL 3: Exclusive Prefix Sum - Algoritmo de Blelloch
//
// Convierte el histograma en CDF (distribucion acumulada).
// EXCLUSIVE significa: cdf[i] = suma de histo[0..i-1], sin incluir histo[i].
// Por definicion: cdf[0] = 0 siempre.
//
// Ejemplo:
//   Histograma: [4, 7, 3, 2]
//   CDF:        [0, 4, 11, 14]
//
// Dos fases sobre shared memory:
//   Up-sweep:   construye sumas parciales (arbol hacia arriba)
//   Down-sweep: propaga valores para producir el scan (arbol hacia abajo)
//
// Complejidad: O(n) trabajo, O(log n) profundidad
// Limitacion: numBins debe ser potencia de 2 y <= 1024 (un solo bloque)
// =============================================================================
__global__
void exclusive_scan(unsigned int* d_data, const size_t n)
{
    extern __shared__ unsigned int temp[];
    int tid = threadIdx.x;
    int offset = 1;

    // Cargar histograma a shared memory
    temp[tid] = (tid < (int)n) ? d_data[tid] : 0;
    __syncthreads();

    // ── UP-SWEEP: construir arbol de sumas parciales ──────────────────────────
    // Al final, temp[n-1] contiene la suma total del histograma
    for (int d = n >> 1; d > 0; d >>= 1) {
        if (tid < d) {
            int ai = offset * (2 * tid + 1) - 1;
            int bi = offset * (2 * tid + 2) - 1;
            temp[bi] += temp[ai];
        }
        offset *= 2;
        __syncthreads();
    }

    // Limpiar el ultimo elemento: esto convierte el inclusive scan en exclusive
    if (tid == 0) temp[n - 1] = 0;
    __syncthreads();

    // ── DOWN-SWEEP: propagar hacia abajo para producir el scan ────────────────
    for (int d = 1; d < (int)n; d *= 2) {
        offset >>= 1;
        if (tid < d) {
            int ai = offset * (2 * tid + 1) - 1;
            int bi = offset * (2 * tid + 2) - 1;
            unsigned int t = temp[ai];
            temp[ai] = temp[bi];  // nodo izquierdo recibe valor del padre
            temp[bi] += t;        // nodo derecho acumula
        }
        __syncthreads();
    }

    // Escribir resultado de vuelta a memoria global
    if (tid < (int)n) d_data[tid] = temp[tid];
}

// =============================================================================
// FUNCION PRINCIPAL
// Orquesta los 4 pasos: min/max -> rango -> histograma -> exclusive scan
// =============================================================================
void your_histogram_and_prefixsum(const float* const d_logLuminance,
                                  unsigned int* const d_cdf,
                                  float &min_logLum,
                                  float &max_logLum,
                                  const size_t numRows,
                                  const size_t numCols,
                                  const size_t numBins)
{
    const size_t n = numRows * numCols;
    const int THREADS = 1024;
    const int BLOCKS  = (n + THREADS - 1) / THREADS;

    // ── PASO 1: Min y Max mediante reduccion paralela ─────────────────────────
    float *d_min_partial, *d_max_partial;
    checkCudaErrors(cudaMalloc(&d_min_partial, sizeof(float) * BLOCKS));
    checkCudaErrors(cudaMalloc(&d_max_partial, sizeof(float) * BLOCKS));

    // shared memory = 2 arrays de THREADS floats (uno para min, uno para max)
    size_t sharedBytes = 2 * THREADS * sizeof(float);
    reduce_min_max<<<BLOCKS, THREADS, sharedBytes>>>(
        d_logLuminance, d_min_partial, d_max_partial, n);
    cudaDeviceSynchronize();

    // Reduccion final de resultados parciales en CPU (BLOCKS es pequeno)
    float* h_min = new float[BLOCKS];
    float* h_max = new float[BLOCKS];
    checkCudaErrors(cudaMemcpy(h_min, d_min_partial, sizeof(float)*BLOCKS, cudaMemcpyDeviceToHost));
    checkCudaErrors(cudaMemcpy(h_max, d_max_partial, sizeof(float)*BLOCKS, cudaMemcpyDeviceToHost));

    min_logLum = h_min[0];
    max_logLum = h_max[0];
    for (int i = 1; i < BLOCKS; i++) {
        min_logLum = min(min_logLum, h_min[i]);
        max_logLum = max(max_logLum, h_max[i]);
    }
    delete[] h_min;
    delete[] h_max;

    // ── PASO 2: Rango ─────────────────────────────────────────────────────────
    float range = max_logLum - min_logLum;

    // ── PASO 3: Histograma ────────────────────────────────────────────────────
    // Inicializar en ceros antes de acumular con atomicAdd
    checkCudaErrors(cudaMemset(d_cdf, 0, sizeof(unsigned int) * numBins));
    histogram_kernel<<<BLOCKS, THREADS>>>(
        d_logLuminance, d_cdf, min_logLum, range, numBins, n);
    cudaDeviceSynchronize();

    // ── PASO 4: Exclusive Scan (Blelloch) ─────────────────────────────────────
    // Un solo bloque procesa todos los bins (numBins <= 1024)
    exclusive_scan<<<1, numBins, numBins * sizeof(unsigned int)>>>(d_cdf, numBins);
    cudaDeviceSynchronize();

    // Liberar memoria temporal de GPU
    checkCudaErrors(cudaFree(d_min_partial));
    checkCudaErrors(cudaFree(d_max_partial));
}


## Celda 5 — Compilación

Compilamos con `nvcc` usando `-arch=sm_75` (GPU T4 de Colab). El Makefile original usaba `sm_20` que ya no es soportado por versiones modernas de CUDA.

Los warnings de comparación de signedness en `loadSaveImage.cpp` son inofensivos — la compilación termina correctamente.

In [ ]:
%cd /content/udacity-cs344-colab/src/HW3
!nvcc -c student_func.cu -O3 -arch=sm_75 -m64
!nvcc -c HW3.cu -I /usr/include/opencv4 -O3 -arch=sm_75 -m64
!g++ -c main.cpp -O3 -Wall -m64 -I /usr/local/cuda/include
!g++ -c loadSaveImage.cpp -I /usr/include/opencv4 -O3 -Wall -m64 -I /usr/local/cuda/include
!g++ -c compare.cpp -I /usr/include/opencv4 -O3 -Wall -m64 -I /usr/local/cuda/include
!g++ -c reference_calc.cpp -I /usr/include/opencv4 -O3 -Wall -m64 -I /usr/local/cuda/include
!nvcc -o HW3 main.o student_func.o HW3.o loadSaveImage.o compare.o reference_calc.o \
    -L /usr/lib/x86_64-linux-gnu \
    -lopencv_core -lopencv_imgproc -lopencv_highgui -lopencv_imgcodecs \
    -O3 -arch=sm_75 -m64
print('Compilacion exitosa')


## Celda 6 — Ejecución

Ejecutamos el binario con la imagen HDR de prueba (`memorial.exr`). El programa aplica el tone mapping y compara el resultado contra la solución de referencia.

- `PASS` = implementación correcta
- El tiempo reportado es el tiempo de ejecución de los kernels CUDA en GPU

In [ ]:
%cd /content/udacity-cs344-colab/src/HW3
!./HW3 memorial.exr


## Celda 7 — Visualización de resultados

Comparamos visualmente las 4 imágenes generadas:

- **memorial_raw:** imagen HDR original sin procesar. Aparece casi completamente negra porque el rango [0–275] no cabe en [0–255] de forma lineal.
- **output:** resultado de nuestro algoritmo CUDA. El CDF redistribuye la luminancia y hace visible toda la escena.
- **reference:** solución de referencia del curso. Debe ser idéntica al output si la implementación es correcta.
- **differenceimage:** diferencia pixel a pixel entre output y reference. Negro = diferencia cero = implementación perfecta.

In [ ]:
import cv2
import matplotlib.pyplot as plt

base = '/content/udacity-cs344-colab/src/HW3/'

imagenes = [
    ('memorial_raw.png',    'Imagen HDR original (sin tone mapping)'),
    ('output.png',          'Output: nuestro tone mapping CUDA'),
    ('reference.png',       'Referencia del curso'),
    ('differenceimage.png', 'Diferencia output vs referencia\n(negro = identicos)'),
]

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
axes = axes.flatten()

for i, (filename, title) in enumerate(imagenes):
    img = cv2.imread(base + filename)
    if img is not None:
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        axes[i].imshow(img_rgb)
    axes[i].set_title(title, fontsize=11)
    axes[i].axis('off')

plt.suptitle('CS344 HW3 - HDR Tone Mapping con CUDA', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print('Resultado: PASS — Implementacion CUDA correcta.')
